# TP N°04 – Named Entity Recognition avec Sequence Models
**Module :** Natural Language Processing  
**Option :** IASD – 2ème année Second Cycle  
**Dataset :** CoNLL-2003  
**Modèles :** BiLSTM, BiLSTM-CRF, GRU

---

## 📦 Installation des dépendances

In [ ]:
!pip install datasets transformers seqeval torch torchcrf scikit-learn matplotlib seaborn -q

## 📚 Importation des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from collections import Counter
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Configuration du device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Utilisation du device : {device}")

# Seed pour la reproductibilité
torch.manual_seed(42)
np.random.seed(42)

---
## 1️⃣ Chargement du Dataset CoNLL-2003

In [ ]:
# Chargement du dataset CoNLL-2003 via Hugging Face
dataset = load_dataset("conll2003", trust_remote_code=True)
print(dataset)

# Affichage des labels NER
label_names = dataset["train"].features["ner_tags"].feature.names
print(f"\n🏷️ Labels NER : {label_names}")
print(f"Nombre de classes : {len(label_names)}")

In [ ]:
# Exploration d'un exemple
example = dataset["train"][0]
print("🔍 Exemple d'entraînement :")
print(f"Tokens  : {example['tokens']}")
print(f"NER tags: {example['ner_tags']}")
print(f"Labels  : {[label_names[t] for t in example['ner_tags']]}")

In [ ]:
# Distribution des entités dans le dataset d'entraînement
all_tags = [tag for sample in dataset["train"] for tag in sample["ner_tags"]]
tag_counts = Counter(all_tags)

plt.figure(figsize=(10, 5))
labels_display = [label_names[k] for k in sorted(tag_counts.keys())]
counts_display = [tag_counts[k] for k in sorted(tag_counts.keys())]
colors = sns.color_palette("Set2", len(labels_display))
plt.bar(labels_display, counts_display, color=colors, edgecolor='black')
plt.title("Distribution des étiquettes NER (CoNLL-2003 Train)", fontsize=14, fontweight='bold')
plt.xlabel("Étiquettes NER")
plt.ylabel("Fréquence")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("distribution_tags.png", dpi=150)
plt.show()
print("\n📊 Distribution :")
for k, v in sorted(tag_counts.items()):
    print(f"  {label_names[k]:15s} : {v:6d} ({v/len(all_tags)*100:.1f}%)")

---
## 2️⃣ Prétraitement des données

In [ ]:
# ─── Construction du vocabulaire ───────────────────────────────────────────────

def build_vocab(dataset_split, min_freq=1):
    """Construit un vocabulaire à partir du dataset."""
    word_freq = Counter()
    for sample in dataset_split:
        # Normalisation de la casse (lowercase)
        tokens = [tok.lower() for tok in sample["tokens"]]
        word_freq.update(tokens)
    
    vocab = {"<PAD>": 0, "<UNK>": 1}  # tokens spéciaux
    for word, freq in word_freq.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

word2idx = build_vocab(dataset["train"], min_freq=1)
idx2word = {v: k for k, v in word2idx.items()}

# Mapping des labels
tag2idx = {tag: idx for idx, tag in enumerate(label_names)}
idx2tag = {v: k for k, v in tag2idx.items()}

print(f"📖 Taille du vocabulaire : {len(word2idx)}")
print(f"🏷️ Nombre de classes NER : {len(tag2idx)}")
print(f"\nMapping des étiquettes : {tag2idx}")

In [ ]:
# ─── Classe Dataset PyTorch ────────────────────────────────────────────────────

class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx):
        self.samples = []
        for sample in data:
            # Tokenization + normalisation de la casse
            tokens = [word2idx.get(tok.lower(), word2idx["<UNK>"]) 
                      for tok in sample["tokens"]]
            tags = sample["ner_tags"]
            self.samples.append((
                torch.tensor(tokens, dtype=torch.long),
                torch.tensor(tags,   dtype=torch.long)
            ))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Padding des séquences pour le batch."""
    tokens_batch, tags_batch = zip(*batch)
    lengths = torch.tensor([len(t) for t in tokens_batch])
    tokens_padded = pad_sequence(tokens_batch, batch_first=True, padding_value=0)
    tags_padded   = pad_sequence(tags_batch,   batch_first=True, padding_value=-1)
    return tokens_padded, tags_padded, lengths


# Création des datasets
train_dataset = NERDataset(dataset["train"],      word2idx, tag2idx)
val_dataset   = NERDataset(dataset["validation"], word2idx, tag2idx)
test_dataset  = NERDataset(dataset["test"],       word2idx, tag2idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"✅ Train   : {len(train_dataset)} phrases")
print(f"✅ Val     : {len(val_dataset)}   phrases")
print(f"✅ Test    : {len(test_dataset)}  phrases")

---
## 3️⃣ Définition des modèles

In [ ]:
# ─── Modèle BiLSTM ────────────────────────────────────────────────────────────

class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags,
                 num_layers=2, dropout=0.3, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_tags)
    
    def forward(self, x, lengths=None):
        embedded = self.dropout(self.embedding(x))
        output, _ = self.lstm(embedded)
        output = self.dropout(output)
        logits = self.fc(output)
        return logits


# ─── Modèle GRU ───────────────────────────────────────────────────────────────

class BiGRU_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags,
                 num_layers=2, dropout=0.3, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.gru = nn.GRU(
            embed_dim, hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_tags)
    
    def forward(self, x, lengths=None):
        embedded = self.dropout(self.embedding(x))
        output, _ = self.gru(embedded)
        output = self.dropout(output)
        logits = self.fc(output)
        return logits


# ─── Modèle BiLSTM + CRF ──────────────────────────────────────────────────────

try:
    from torchcrf import CRF
    CRF_AVAILABLE = True
except ImportError:
    CRF_AVAILABLE = False
    print("⚠️ torchcrf non disponible. Le modèle BiLSTM-CRF sera ignoré.")

if CRF_AVAILABLE:
    class BiLSTM_CRF_NER(nn.Module):
        def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags,
                     num_layers=2, dropout=0.3, padding_idx=0):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
            self.lstm = nn.LSTM(
                embed_dim, hidden_dim // 2,
                num_layers=num_layers,
                bidirectional=True,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0
            )
            self.dropout = nn.Dropout(dropout)
            self.fc  = nn.Linear(hidden_dim, num_tags)
            self.crf = CRF(num_tags, batch_first=True)
        
        def forward(self, x, tags=None, mask=None):
            embedded = self.dropout(self.embedding(x))
            output, _ = self.lstm(embedded)
            output = self.dropout(output)
            emissions = self.fc(output)
            if tags is not None:  # mode entraînement
                loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
                return loss
            return self.crf.decode(emissions, mask=mask)  # mode inférence

print("✅ Modèles définis : BiLSTM, BiGRU" + (", BiLSTM-CRF" if CRF_AVAILABLE else ""))

---
## 4️⃣ Fonctions d'entraînement et d'évaluation

In [ ]:
# ─── Fonctions utilitaires ─────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device, is_crf=False):
    model.train()
    total_loss = 0
    for tokens, tags, lengths in loader:
        tokens, tags = tokens.to(device), tags.to(device)
        optimizer.zero_grad()
        
        if is_crf:
            mask = (tags != -1)
            tags_crf = tags.clone()
            tags_crf[tags_crf == -1] = 0
            loss = model(tokens, tags=tags_crf, mask=mask)
        else:
            logits = model(tokens)
            # Reshape pour CrossEntropy : (B*T, C)
            B, T, C = logits.shape
            loss = criterion(logits.view(B * T, C), tags.view(B * T))
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device, idx2tag, is_crf=False):
    model.eval()
    all_preds, all_true = [], []
    
    with torch.no_grad():
        for tokens, tags, lengths in loader:
            tokens, tags = tokens.to(device), tags.to(device)
            
            if is_crf:
                mask = (tags != -1)
                preds = model(tokens, mask=mask)  # list of lists
                for pred_seq, true_seq, m in zip(preds, tags, mask):
                    true_labels = true_seq[m].tolist()
                    all_preds.append([idx2tag[p] for p in pred_seq])
                    all_true.append([idx2tag[t] for t in true_labels])
            else:
                logits = model(tokens)
                preds  = logits.argmax(dim=-1)   # (B, T)
                for pred_seq, true_seq in zip(preds, tags):
                    mask_i    = true_seq != -1
                    pred_list = pred_seq[mask_i].tolist()
                    true_list = true_seq[mask_i].tolist()
                    all_preds.append([idx2tag[p] for p in pred_list])
                    all_true.append([idx2tag[t] for t in true_list])
    
    f1  = f1_score(all_true, all_preds)
    prec = precision_score(all_true, all_preds)
    rec  = recall_score(all_true, all_preds)
    return f1, prec, rec, all_preds, all_true


def train_model(model, train_loader, val_loader, optimizer, criterion,
                device, idx2tag, num_epochs=10, is_crf=False, model_name="Model"):
    history = {"train_loss": [], "val_f1": [], "val_prec": [], "val_rec": []}
    best_f1, best_state = 0, None
    
    print(f"\n🚀 Entraînement : {model_name}")
    print("-" * 60)
    
    for epoch in range(1, num_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, is_crf)
        val_f1, val_prec, val_rec, _, _ = evaluate(model, val_loader, device, idx2tag, is_crf)
        
        history["train_loss"].append(train_loss)
        history["val_f1"].append(val_f1)
        history["val_prec"].append(val_prec)
        history["val_rec"].append(val_rec)
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f"Époque {epoch:02d}/{num_epochs} | Loss: {train_loss:.4f} | "
              f"Val F1: {val_f1:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f}")
    
    # Chargement du meilleur modèle
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"\n✅ Meilleur F1 validation : {best_f1:.4f}")
    return history

print("✅ Fonctions d'entraînement définies.")

---
## 5️⃣ Entraînement – Modèle BiLSTM

In [ ]:
# ─── Hyperparamètres ──────────────────────────────────────────────────────────
VOCAB_SIZE = len(word2idx)
EMBED_DIM  = 128
HIDDEN_DIM = 256
NUM_TAGS   = len(tag2idx)
NUM_LAYERS = 2
DROPOUT    = 0.3
NUM_EPOCHS = 10
LR         = 1e-3

# Poids pour la CrossEntropy (ignorer le padding)
criterion = nn.CrossEntropyLoss(ignore_index=-1)

# ─── BiLSTM ───────────────────────────────────────────────────────────────────
bilstm_model = BiLSTM_NER(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM, num_tags=NUM_TAGS,
    num_layers=NUM_LAYERS, dropout=DROPOUT
).to(device)

optimizer_lstm = optim.Adam(bilstm_model.parameters(), lr=LR, weight_decay=1e-5)
scheduler_lstm = optim.lr_scheduler.StepLR(optimizer_lstm, step_size=3, gamma=0.5)

print(f"📐 Paramètres BiLSTM : {sum(p.numel() for p in bilstm_model.parameters()):,}")

history_lstm = train_model(
    bilstm_model, train_loader, val_loader,
    optimizer_lstm, criterion, device, idx2tag,
    num_epochs=NUM_EPOCHS, is_crf=False, model_name="BiLSTM"
)

## 6️⃣ Entraînement – Modèle BiGRU

In [ ]:
bigru_model = BiGRU_NER(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM, num_tags=NUM_TAGS,
    num_layers=NUM_LAYERS, dropout=DROPOUT
).to(device)

optimizer_gru = optim.Adam(bigru_model.parameters(), lr=LR, weight_decay=1e-5)

print(f"📐 Paramètres BiGRU : {sum(p.numel() for p in bigru_model.parameters()):,}")

history_gru = train_model(
    bigru_model, train_loader, val_loader,
    optimizer_gru, criterion, device, idx2tag,
    num_epochs=NUM_EPOCHS, is_crf=False, model_name="BiGRU"
)

## 7️⃣ Entraînement – Modèle BiLSTM-CRF (si disponible)

In [ ]:
history_crf = None

if CRF_AVAILABLE:
    bilstm_crf_model = BiLSTM_CRF_NER(
        vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM, num_tags=NUM_TAGS,
        num_layers=NUM_LAYERS, dropout=DROPOUT
    ).to(device)
    
    optimizer_crf = optim.Adam(bilstm_crf_model.parameters(), lr=LR, weight_decay=1e-5)
    
    print(f"📐 Paramètres BiLSTM-CRF : {sum(p.numel() for p in bilstm_crf_model.parameters()):,}")
    
    history_crf = train_model(
        bilstm_crf_model, train_loader, val_loader,
        optimizer_crf, criterion, device, idx2tag,
        num_epochs=NUM_EPOCHS, is_crf=True, model_name="BiLSTM-CRF"
    )
else:
    print("⚠️ BiLSTM-CRF ignoré (torchcrf non installé).")

---
## 8️⃣ Visualisation des courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, NUM_EPOCHS + 1)

# Loss d'entraînement
axes[0].plot(epochs, history_lstm["train_loss"], 'b-o', label="BiLSTM", linewidth=2)
axes[0].plot(epochs, history_gru["train_loss"],  'r-s', label="BiGRU",  linewidth=2)
if history_crf:
    axes[0].plot(epochs, history_crf["train_loss"], 'g-^', label="BiLSTM-CRF", linewidth=2)
axes[0].set_title("Loss d'entraînement", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 validation
axes[1].plot(epochs, history_lstm["val_f1"], 'b-o', label="BiLSTM", linewidth=2)
axes[1].plot(epochs, history_gru["val_f1"],  'r-s', label="BiGRU",  linewidth=2)
if history_crf:
    axes[1].plot(epochs, history_crf["val_f1"], 'g-^', label="BiLSTM-CRF", linewidth=2)
axes[1].set_title("F1-Score Validation", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Époque")
axes[1].set_ylabel("F1-Score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("courbes_apprentissage.png", dpi=150)
plt.show()

---
## 9️⃣ Évaluation sur le jeu de test

In [ ]:
print("=" * 65)
print("📊 RÉSULTATS SUR LE JEU DE TEST")
print("=" * 65)

results = {}

# BiLSTM
f1, prec, rec, preds_lstm, true_lstm = evaluate(
    bilstm_model, test_loader, device, idx2tag, is_crf=False)
results["BiLSTM"] = {"F1": f1, "Précision": prec, "Rappel": rec}
print(f"\n🔵 BiLSTM     → F1: {f1:.4f} | Précision: {prec:.4f} | Rappel: {rec:.4f}")

# BiGRU
f1, prec, rec, preds_gru, true_gru = evaluate(
    bigru_model, test_loader, device, idx2tag, is_crf=False)
results["BiGRU"] = {"F1": f1, "Précision": prec, "Rappel": rec}
print(f"🔴 BiGRU      → F1: {f1:.4f} | Précision: {prec:.4f} | Rappel: {rec:.4f}")

# BiLSTM-CRF
if CRF_AVAILABLE:
    f1, prec, rec, preds_crf, true_crf = evaluate(
        bilstm_crf_model, test_loader, device, idx2tag, is_crf=True)
    results["BiLSTM-CRF"] = {"F1": f1, "Précision": prec, "Rappel": rec}
    print(f"🟢 BiLSTM-CRF → F1: {f1:.4f} | Précision: {prec:.4f} | Rappel: {rec:.4f}")

print("\n" + "=" * 65)

In [ ]:
# Rapport détaillé par classe (BiLSTM)
print("\n📋 Rapport détaillé – BiLSTM :")
print(classification_report(true_lstm, preds_lstm))

In [ ]:
# Rapport détaillé par classe (BiGRU)
print("\n📋 Rapport détaillé – BiGRU :")
print(classification_report(true_gru, preds_gru))

if CRF_AVAILABLE:
    print("\n📋 Rapport détaillé – BiLSTM-CRF :")
    print(classification_report(true_crf, preds_crf))

In [ ]:
# Tableau comparatif
df_results = pd.DataFrame(results).T.round(4)
print("\n📊 Tableau Comparatif des Modèles :")
print(df_results.to_string())

# Visualisation
df_results.plot(kind='bar', figsize=(9, 5), colormap='Set2', edgecolor='black')
plt.title("Comparaison des modèles NER sur CoNLL-2003 Test", fontsize=13, fontweight='bold')
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig("comparaison_modeles.png", dpi=150)
plt.show()

---
## 🔟 Inférence – Prédiction sur des phrases personnalisées

In [ ]:
def predict_ner(sentence, model, word2idx, idx2tag, device, is_crf=False):
    """Prédit les entités nommées dans une phrase."""
    tokens = sentence.split()
    indices = [word2idx.get(tok.lower(), word2idx["<UNK>"]) for tok in tokens]
    x = torch.tensor([indices], dtype=torch.long).to(device)
    
    model.eval()
    with torch.no_grad():
        if is_crf:
            preds = model(x)[0]
            pred_tags = [idx2tag[p] for p in preds]
        else:
            logits = model(x)
            preds  = logits.argmax(dim=-1)[0].tolist()
            pred_tags = [idx2tag[p] for p in preds]
    
    print(f"\n📝 Phrase : {sentence}")
    print(f"{'Token':<20} {'Étiquette':<15}")
    print("-" * 35)
    for tok, tag in zip(tokens, pred_tags):
        marker = "  ◀" if tag != "O" else ""
        print(f"{tok:<20} {tag:<15}{marker}")


# Tests
test_sentences = [
    "Barack Obama was born in Hawaii and studied at Harvard University .",
    "Google is headquartered in Mountain View California .",
    "The European Union and United Nations met in Paris last week .",
    "Apple CEO Tim Cook announced new products in New York ."
]

print("=" * 60)
print("🔍 PRÉDICTIONS BiLSTM")
print("=" * 60)
for s in test_sentences:
    predict_ner(s, bilstm_model, word2idx, idx2tag, device, is_crf=False)

---
## 1️⃣1️⃣ Amélioration du modèle
### Techniques appliquées :
1. **Dropout** – régularisation
2. **Gradient Clipping** – stabilité de l'entraînement
3. **Learning Rate Scheduling** – ajustement du LR
4. **Weight Decay** – régularisation L2
5. **BiLSTM-CRF** – modélisation des dépendances entre labels

### Modèle amélioré (BiLSTM avec plus de couches)

In [ ]:
# Modèle amélioré : plus de couches LSTM + embed_dim plus grand
improved_model = BiLSTM_NER(
    vocab_size=VOCAB_SIZE,
    embed_dim=200,        # embedding plus large
    hidden_dim=512,       # hidden plus large
    num_tags=NUM_TAGS,
    num_layers=3,         # 3 couches LSTM
    dropout=0.4
).to(device)

optimizer_imp = optim.AdamW(improved_model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_imp = optim.lr_scheduler.CosineAnnealingLR(optimizer_imp, T_max=NUM_EPOCHS)

print(f"📐 Paramètres Modèle Amélioré : {sum(p.numel() for p in improved_model.parameters()):,}")

history_imp = train_model(
    improved_model, train_loader, val_loader,
    optimizer_imp, criterion, device, idx2tag,
    num_epochs=NUM_EPOCHS, is_crf=False, model_name="BiLSTM Amélioré"
)

f1, prec, rec, _, _ = evaluate(improved_model, test_loader, device, idx2tag)
print(f"\n🏆 BiLSTM Amélioré → F1: {f1:.4f} | Précision: {prec:.4f} | Rappel: {rec:.4f}")

---
## 1️⃣2️⃣ Sauvegarde des modèles

In [ ]:
import os
os.makedirs("saved_models", exist_ok=True)

# Sauvegarde
torch.save({
    'model_state_dict': bilstm_model.state_dict(),
    'word2idx': word2idx,
    'idx2tag': idx2tag,
    'config': {'embed_dim': EMBED_DIM, 'hidden_dim': HIDDEN_DIM,
               'num_layers': NUM_LAYERS, 'dropout': DROPOUT}
}, "saved_models/bilstm_ner.pt")

torch.save({
    'model_state_dict': bigru_model.state_dict(),
    'word2idx': word2idx,
    'idx2tag': idx2tag,
}, "saved_models/bigru_ner.pt")

torch.save({
    'model_state_dict': improved_model.state_dict(),
    'word2idx': word2idx,
    'idx2tag': idx2tag,
}, "saved_models/bilstm_improved_ner.pt")

print("✅ Modèles sauvegardés dans ./saved_models/")
for f in os.listdir("saved_models"):
    size = os.path.getsize(f"saved_models/{f}") / 1e6
    print(f"   {f:30s} — {size:.1f} MB")

---
## 📝 Résumé et Conclusion

Dans ce TP, nous avons implémenté et comparé plusieurs architectures de réseaux de neurones récurrents pour la reconnaissance d'entités nommées (NER) sur le dataset CoNLL-2003 :

| Modèle | Description | Avantages |
|--------|-------------|----------|
| **BiLSTM** | LSTM bidirectionnel 2 couches | Contexte passé et futur |
| **BiGRU** | GRU bidirectionnel 2 couches | Plus rapide, moins de paramètres |
| **BiLSTM-CRF** | BiLSTM + CRF layer | Dépendances entre labels |
| **BiLSTM Amélioré** | 3 couches, embed=200 | Meilleures performances |

**Métriques évaluées :** Précision, Rappel, F1-Score (via seqeval)

**Entités cibles :** PER (Personne), ORG (Organisation), LOC (Lieu), MISC (Divers)

**Améliorations possibles :**
- Utiliser des embeddings pré-entraînés (GloVe, fastText)
- Intégrer des embeddings de caractères
- Utiliser BERT/CamemBERT pour du transfer learning
- Data augmentation